In [7]:
display(config_box)
display(launch_button)
display(output_area)
# Bienvenue cliquez ICI en 2 après !!!! :) 

Button(button_style='success', description='▶ LANCER SIMULATION', icon='play', style=ButtonStyle())

Output(outputs=({'output_type': 'stream', 'text': '\n=========================================================…

In [1]:

# 1er clique
#--------------------------
# 2026 06 13 version v9.12 ************************************
# *
# *   ███████╗██████╗ ██╗  ██╗██╗██████╗ ██╗████████╗
# *   ██╔════╝██╔══██╗██║  ██║██║██╔══██╗██║╚══██╔══╝
# *   ███████╗██████╔╝███████║██║██████╔╝██║   ██║
# *   ╚════██║██╔═══╝ ██╔══██║██║██╔══██╗██║   ██║
# *   ███████║██║     ██║  ██║██║██║  ██║██║   ██║
# *   ╚══════╝╚═╝     ╚═╝  ╚═╝╚═╝╚═╝  ╚═╝╚═╝   ╚═╝
# *
# *   Smooth Particle Hydrodynamics
# *   Integrated Resource for Instruction and Teaching
# *
# *   Educational & Research CFD Platform
# *
# ****************************************************************/
#****************************************************************
# Masse conservation (no variation) and no volume variation
# ALE formulation
# Ordre 2 and 3 in time and space with MLS and Renormalization for div operator 
# Six tests cas  : 
#                   jet impact, impact of a column of water on a plate or deux jets impacted
#                   funnel , vsicosity study via water release Funnel Discharge Benchmark (FDB)
#                   DamBreak, ALE, L dambreak
#                   TGV, : ALE,El L vortex preservation
#                   car (FSI)  : full lagrangian interaction between a car (VBL) and a water tank
#                   floating body  https://www.spheric-sph.org/tests/test-12
#                   Test mesh mouvement (no Euler or Navier-Stokes resolution only mesh displacement) : based on TGV
#                   Poiseuille
# ---- TO-DO--------------
#  OPtimisation des calculs (vitesse, doublon etc)
#  protocole funnel faire avec Baptiste
#  faire Poiseuille convergence log-log
#  faire convergence TGV eulerian, lagrangian log-log
#  shifting ++
#  theorical aspect
# --------------------------  
# Structure du code
#1. Paramètres physiques et numériques per testcase
#2. Fonctions SPH (kernel, gradient, pression, densité, Riemann)
#3. Fonctions ALE (entropie, v_mesh)
#4. Initialisation maillage + particules
#5. Boucle temporelle :
  # a) Mise à jour conditions aux frontières fantômes
 #  b) Calculs Shepard ou RENORM / ALE / v_mesh
 #  c) Calculs flux massiques et accélérations
 #  d) Mise à jour rho, vel, pos
 #  e) Calcul dt CFL
  # f) Sauvegarde VTK
# ------------------------
# Biblio 
#test case TGV and jet2D in https://www.sciencedirect.com/science/article/pii/S0045782523002839
#test case DamBreak  https://www.spheric-sph.org/tests/test-02 et https://www.tandfonline.com/doi/full/10.1080/21664250.2019.1672124?utm_source=chatgpt.com#d1e3653
#test case funnel https://cdn.standards.iteh.ai/samples/73851/ccc06afc20a046ac83a46e30264f221c/ISO-2431-2019.pdf
#test case car
#test case Poiseuille 
from IPython.display import HTML, Javascript, display as ipy_display
import numpy as np
import os
from scipy.spatial import cKDTree
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.spatial import Voronoi
# Cellule 1 : Widgets uniquement
import ipywidgets as widgets
from IPython.display import display, HTML

# ============ INITIALISER ============
testCase = 'DamBreak'
mesh_mode = 'lagrangian'
mode_ale = 'wass'
beta_geom = 0.001
alpha = 0.01
mode_p_refinement = 0
mode_RK = 2
Kernel = True
Relax = True
RenormON = False
ShepardCorr = False
ViscoONOFF = False
ShiftPart = False
ShiftMethod = 'delta_sph'  # 'delta_sph' ou 'lloyd'
ShiftCFL = 0.05            # Coefficient stabilité (0.01-0.1)
ShiftR = 0.2               # Ratio remplissage (0.1-0.3)
ShiftBetaLloyd = 0.5       # Coefficient Lloyd
Filtre = False
randompos = True

ModeDEBUG = False
ModeGEOM = False
CFL = 0.1
dx = 1
h = 2.0
C0 = 7.0
# ============ WIDGETS ============
testCase_dropdown = widgets.Dropdown(
    options={
        'DamBreak': 'DamBreak',
        'TGV': 'TGV',
        'Poiseuille': 'Poiseuille',
        'TGVmesh': 'TGVmesh',
        'Jet': 'jet',
        'Funnel': 'funnel',
        'Car': 'car',
        'BodyF': 'bodyF',
        'AirFoil': 'AirFoil',
        'Mill': 'Mill',
    },
    value='DamBreak',
    description='Test Case:',
    style={'description_width': '150px'}
)

mesh_mode_dropdown = widgets.Dropdown(
    options={'Lagrangian': 'lagrangian', 'Eulerian': 'eulerian', 'ALE': 'ale'},
    value='lagrangian',
    description='Mesh Mode:',
    style={'description_width': '150px'}
)

mode_ale_dropdown = widgets.Dropdown(
    options={'Lloyd': 'lloyd', 'Wass': 'wass', 'Shepard': 'shepard', 'Coupled': 'coupled'},
    value='wass',
    description='ALE Method:',
    style={'description_width': '150px'}
)

mode_p_refinement_dropdown = widgets.Dropdown(
    options={'O0': 0, 'O1 (MLS)': 1, 'O2 (Quad)': 2},
    value=0,
    description='Spatial Order:',
    style={'description_width': '150px'}
)

mode_RK_dropdown = widgets.Dropdown(
    options={'Euler': 1, 'RK2': 2, 'RK3': 3},
    value=1,
    description='Time Integration:',
    style={'description_width': '150px'}
)

beta_geom_slider = widgets.FloatSlider(
    value=0.001, min=0.0001, max=0.1, step=0.001,
    description='ALE Method β:',
    style={'description_width': '150px'}
)

alpha_slider = widgets.FloatSlider(
    value=0.01, min=0.0, max=10.0, step=0.1,
    description='ALE Method α:',
    style={'description_width': '150px'}
)

CFL_slider = widgets.FloatSlider(
    value=0.1, min=0.01, max=0.5, step=0.01,
    description='CFL:',
    style={'description_width': '150px'}
)

dx_slider = widgets.FloatSlider(
    value=1, min=1, max=100, step=1,
    description='Resolution:',
    style={'description_width': '150px'}
)
h_slider = widgets.FloatSlider(
    value=2.0, min=1.2, max=2.5, step=0.1,
    description='h/dx:',
    style={'description_width': '150px'}
)
C0_slider = widgets.FloatSlider(
    value=7.0, min=0.0, max=20.0, step=2.0,
    description='C0/Umax:',
    style={'description_width': '150px'}
)
ShiftPart_toggle = widgets.Checkbox(value=False, description='✓ Particle Shifting', indent=False)
ShiftMethod_dropdown = widgets.Dropdown(
    options={'δ-SPH': 'delta_sph', 'Lloyd': 'lloyd'},
    value='delta_sph',
    description='Shift Method:',
    style={'description_width': '150px'}
)
ShiftCFL_slider = widgets.FloatSlider(
    value=0.05, min=0.01, max=0.2, step=0.01,
    description='Shift CFL:',
    style={'description_width': '150px'}
)
ShiftR_slider = widgets.FloatSlider(
    value=0.2, min=0.05, max=0.5, step=0.05,
    description='Shift R:',
    style={'description_width': '150px'}
)

Kernel_toggle = widgets.Checkbox(value=True, description='✓ Kernel C2', indent=False)
RenormON_toggle = widgets.Checkbox(value=False, description='✓ Renormalisation', indent=False)
Relax_toggle = widgets.Checkbox(value=False, description='✓ Relaxation mesh', indent=False)
ShepardCorr_toggle = widgets.Checkbox(value=False, description='✓ Shepard Correction', indent=False)
ViscoONOFF_toggle = widgets.Checkbox(value=False, description='✓ Viscosité (Morris)', indent=False)
ShiftPart_toggle = widgets.Checkbox(value=False, description='✓ Particle Shifting', indent=False)
Filtre_toggle = widgets.Checkbox(value=False, description='✓ Filtre Densité', indent=False)
randompos_toggle = widgets.Checkbox(value=True, description='✓ Bruit Position', indent=False)
ModeDEBUG_toggle = widgets.Checkbox(value=False, description='✓ Mode DEBUG', indent=False)
ModeGEOM_toggle = widgets.Checkbox(value=False, description='✓ Mode GEOM', indent=False)

# ============ OUTOUT ZONE (Pour intercepter les prints) ============
output_area = widgets.Output()

# ============ CALLBACK ============
def on_launch_clicked(b):
    global testCase, mesh_mode, mode_ale, mode_p_refinement, mode_RK
    global beta_geom, alpha, CFL, dx, h, C0
    global Kernel,RenormON,Relax, ShepardCorr, ViscoONOFF, ShiftPart, Filtre, randompos, ModeDEBUG, ModeGEOM,ShiftMethod,ShiftCFL,ShiftR,ShiftBetaLloyd
    
    # On vide la zone d'affichage des anciennes simulations avant de lancer la nouvelle
    output_area.clear_output(wait=False)
    
    # On capture tout ce qui se passe dans ce bloc pour l'afficher dans output_area
    with output_area:
        testCase = testCase_dropdown.value
        mesh_mode = mesh_mode_dropdown.value
        
        mode_p_refinement = mode_p_refinement_dropdown.value
        mode_RK = mode_RK_dropdown.value
        beta_geom = beta_geom_slider.value
        alpha = alpha_slider.value
        CFL = CFL_slider.value
        dx = dx_slider.value
        h = h_slider.value
        C0 = C0_slider.value
        Kernel =Kernel_toggle.value
        RenormON = RenormON_toggle.value
        Relax = Relax_toggle.value
        ShepardCorr = ShepardCorr_toggle.value
        ViscoONOFF = ViscoONOFF_toggle.value
        ShiftPart = ShiftPart_toggle.value
        Filtre = Filtre_toggle.value
        randompos = randompos_toggle.value
        ModeDEBUG = ModeDEBUG_toggle.value
        ModeGEOM = ModeGEOM_toggle.value
        mode_ale = mode_ale_dropdown.value
        ShiftPart = ShiftPart_toggle.value
        ShiftMethod = ShiftMethod_dropdown.value
        ShiftCFL = ShiftCFL_slider.value
        ShiftR = ShiftR_slider.value
        ShiftBetaLloyd = ShiftBetaLloyd  # Valeur fixe
        
        print("\n" + "="*70)
        print("✓ CONFIG APPLIQUÉE - LANCEMENT SIMULATION")
        print("="*70)
        print(f"Test: {testCase} | Mode: {mesh_mode} | ALE: {mode_ale}")
        print(f"Order: {mode_p_refinement} | RK{mode_RK} | CFL={CFL} | Resolution={dx}| h/dx={h}| C0/Umax={C0}")
        print(f"KernelC2: {Kernel} | Renorm: {RenormON} | Relax : {Relax}| Visco: {ViscoONOFF} | DEBUG: {ModeDEBUG} | DEBUG GEOM: {ModeGEOM}")
        # Afficher configuration
        print(f"Shifting: {ShiftPart} | Method: {ShiftMethod} | CFL={ShiftCFL} | R={ShiftR}")
        print("="*70 + "\n")
        
        # ===== EXÉCUTE SIMULATION.IPYNB =====
        %run Simulation.ipynb

launch_button = widgets.Button(
    description='▶ LANCER SIMULATION',
    button_style='success',
    icon='play'
)
launch_button.on_click(on_launch_clicked)

# ============ AFFICHER ============
config_box = widgets.VBox([
    widgets.HTML("<h3> 🫧 🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧 </h3>"),
    widgets.HTML("<h3> 🫧       💧SPHIRIT v9.12 LTS 🌊      🫧 </h3>"),
    widgets.HTML("<h3> 🫧 🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧🫧 </h3>"),
    widgets.HTML("<h3>📋 Configuration</h3>"),
    testCase_dropdown,
    mesh_mode_dropdown,
    
    mode_p_refinement_dropdown,
    mode_RK_dropdown,
    widgets.HTML("<h3>⚙️ Paramètres ⚙️</h3>"),    
    CFL_slider,
    dx_slider,
    h_slider,
    C0_slider,
    widgets.HTML("<h3>✓ 📐 Tests Codes et Maillage  🕸️ </h3>"),
    ModeDEBUG_toggle,
    ModeGEOM_toggle,
    widgets.HTML("<h3>✓ ⚛️ Options physiques 🧠 </h3>"),
    widgets.HBox([ ViscoONOFF_toggle]),
    widgets.HTML("<h3>✓ 🧮 Options numériques ∑ </h3>"),
    widgets.HBox([Kernel_toggle ]),
    widgets.HBox([RenormON_toggle,Relax_toggle, ShepardCorr_toggle]),
    widgets.HBox([ShiftPart_toggle, Filtre_toggle, randompos_toggle]),    
    widgets.HTML("<h3>✓ 🌀 Options ALE ➿ </h3>"),
    mode_ale_dropdown,
    widgets.HBox([ShiftMethod_dropdown]),
    ShiftCFL_slider,
    ShiftR_slider,
    beta_geom_slider,
    alpha_slider,
])

display(config_box)
display(launch_button)
# Pensez bien à afficher la zone d'output en dessous du bouton !
display(output_area)



Button(button_style='success', description='▶ LANCER SIMULATION', icon='play', style=ButtonStyle())

Output()